In [6]:
import sqlite3
import pandas as pd
import os


def verifier_coherence(csv_path, db_path):
    df = pd.read_csv(csv_path)
    conn = sqlite3.connect(db_path)

    print("=== 1. CONTRÔLE DE VOLUMÉTRIE ===")
    nb_clients_csv = len(df)
    nb_clients_sql = conn.execute("SELECT COUNT(*) FROM client").fetchone()[0]

    nb_hist_csv = len(df) * 6
    nb_hist_sql = conn.execute(
        "SELECT COUNT(*) FROM historique_mensuel"
    ).fetchone()[0]

    print(f"Clients     -> CSV : {nb_clients_csv:,} | BDD : {nb_clients_sql:,}")
    print(f"Historique  -> CSV : {nb_hist_csv:,} | BDD : {nb_hist_sql:,}")
    assert (
        nb_clients_csv == nb_clients_sql
    ), "❌ Écart détecté sur le nombre de clients !"
    assert (
        nb_hist_csv == nb_hist_sql
    ), "❌ Écart détecté sur le nombre d'historiques !"

    print("\n=== 2. TOTAUX DE CONTRÔLE (CHECKSUMS) ===")
    # Somme des plafonds de crédit
    sum_plafond_csv = df["LIMIT_BAL"].sum()
    sum_plafond_sql = conn.execute("SELECT SUM(plafond) FROM client").fetchone()[
        0
    ]

    # Somme totale des en-cours (BILL_AMT1 à BILL_AMT6)
    sum_encours_csv = sum(df[f"BILL_AMT{i}"].sum() for i in range(1, 7))
    sum_encours_sql = conn.execute(
        "SELECT SUM(montant_encours) FROM historique_mensuel"
    ).fetchone()[0]

    # Distribution de la cible (defaut)
    target_col = "dpnm" if "dpnm" in df.columns else "default_payment_next_month"
    defauts_csv = df[target_col].sum()
    defauts_sql = conn.execute(
        "SELECT SUM(code_statut_defaut) FROM client"
    ).fetchone()[0]

    print(
        f"Somme Plafonds  -> CSV : {sum_plafond_csv:,} | BDD : {sum_plafond_sql:,}"
    )
    print(
        f"Somme En-cours  -> CSV : {sum_encours_csv:,} | BDD : {sum_encours_sql:,}"
    )
    print(
        f"Total Défauts   -> CSV : {defauts_csv:,} | BDD : {defauts_sql:,}"
    )

    assert (
        sum_plafond_csv == sum_plafond_sql
    ), "❌ Écart détecté sur la somme des plafonds !"
    assert (
        sum_encours_csv == sum_encours_sql
    ), "❌ Écart détecté sur la somme des encours !"
    assert (
        defauts_csv == defauts_sql
    ), "❌ Écart détecté sur le nombre de défauts !"

    print("\n=== 3. SONDAGE PONCTUEL (3 CLIENTS ALÉATOIRES) ===")
    target_col = "dpnm" if "dpnm" in df.columns else "default_payment_next_month"

    # Tirage au sort de 3 clients (random_state optionnel pour fixer le tirage)
    sample_clients = df.sample(n=3, random_state=42)

    for _, row_csv in sample_clients.iterrows():
        client_id = int(row_csv["ID"])

        # Extraction des données client depuis SQL
        client_sql = conn.execute(
            "SELECT client_id, age, plafond, code_statut_defaut FROM client WHERE client_id = ?",
            (client_id,),
        ).fetchone()

        print(f"\n--- Client ID : {client_id} ---")
        print(
            f"  CSV ➔ Âge: {row_csv['AGE']} | Plafond: {row_csv['LIMIT_BAL']:,} | Cible: {row_csv[target_col]}"
        )
        print(
            f"  BDD ➔ Âge: {client_sql[1]} | Plafond: {client_sql[2]:,} | Cible: {client_sql[3]}"
        )

        # Assertions de contrôle automatique
        assert client_sql[1] == int(
            row_csv["AGE"]
        ), f"Incohérence âge pour client {client_id}"
        assert client_sql[2] == int(
            row_csv["LIMIT_BAL"]
        ), f"Incohérence plafond pour client {client_id}"
        assert client_sql[3] == int(
            row_csv[target_col]
        ), f"Incohérence cible pour client {client_id}"

    print("\n✓ Les 3 clients tirés au sort correspondent parfaitement.")

    conn.close()
    print("\n✅ TOUS LES TESTS DE COHÉRENCE SONT VALIDÉS AVEC SUCCÈS !")


# Le chemin ABSOLU du dossier où se trouve ce script
script_dir = os.getcwd()

# Remonter à la racine du projet (un niveau au-dessus de src/)
root_dir = os.path.dirname(script_dir)

# Construire les chemins vers tes fichiers
csv_path = os.path.join(root_dir, 'data', 'creditcard_pret_ingestion.csv')
db_path = os.path.join(root_dir, 'database', 'creditcard.db')

# Exécution
verifier_coherence(csv_path, db_path)

=== 1. CONTRÔLE DE VOLUMÉTRIE ===
Clients     -> CSV : 30,000 | BDD : 30,000
Historique  -> CSV : 180,000 | BDD : 180,000

=== 2. TOTAUX DE CONTRÔLE (CHECKSUMS) ===
Somme Plafonds  -> CSV : 5,024,529,680 | BDD : 5,024,529,680
Somme En-cours  -> CSV : 8,095,850,136 | BDD : 8,095,850,136
Total Défauts   -> CSV : 6,636 | BDD : 6,636

=== 3. SONDAGE PONCTUEL (3 CLIENTS ALÉATOIRES) ===

--- Client ID : 2309 ---
  CSV ➔ Âge: 25 | Plafond: 30,000 | Cible: 0
  BDD ➔ Âge: 25 | Plafond: 30,000 | Cible: 0

--- Client ID : 22405 ---
  CSV ➔ Âge: 26 | Plafond: 150,000 | Cible: 0
  BDD ➔ Âge: 26 | Plafond: 150,000 | Cible: 0

--- Client ID : 23398 ---
  CSV ➔ Âge: 32 | Plafond: 70,000 | Cible: 0
  BDD ➔ Âge: 32 | Plafond: 70,000 | Cible: 0

✓ Les 3 clients tirés au sort correspondent parfaitement.

✅ TOUS LES TESTS DE COHÉRENCE SONT VALIDÉS AVEC SUCCÈS !
